## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
## add your code hereimport sys

# 增加递归深度，以防深层分治报错
sys.setrecursionlimit(2000)

def solve():
    # 快速 IO，一次性读入所有数据
    input_data = sys.stdin.read().split()
    if not input_data:
        return
    
    n = int(input_data[0])
    A = int(input_data[1])
    B = int(input_data[2])
    p = [int(x) for x in input_data[3:3+n]]
    
    answer = []
    
    # 魔法操作实现
    def useSwapMagic():
        answer.append(0)
        for i in range(n):
            if p[i] == A: p[i] = B
            elif p[i] == B: p[i] = A

    def useAddMagic(v):
        v %= n
        if v < 0: v += n
        if v == 0: return
        answer.append(v)
        for i in range(n):
            p[i] = (p[i] + v) % n

    def useXorMagic(v):
        if v == 0: return
        answer.append(-v)
        for i in range(n):
            p[i] ^= v

    # 计算低位长度 lowbitLen
    lowbitLen = (A - B + n) % n
    lowbitLen &= -lowbitLen
    if lowbitLen == 0:
        lowbitLen = n

    def getPairPos(x, y):
        delta = (y - x + n - lowbitLen + n) % n
        px = 0
        py = 0
        step = n // 2
        while step >= 2 * lowbitLen:
            if delta >= step:
                delta -= step
                py += step // 2
            else:
                px += step // 2
            step >>= 1
            
        px += n // 2
        px += (x & (lowbitLen - 1))
        py += (x & (lowbitLen - 1))
        return px, py

    def applySwap(x, y):
        # 如果需要借助中间元素进行中继交换
        if (x // lowbitLen) % 2 == (y // lowbitLen) % 2:
            if (x // lowbitLen) % 2 == 0:
                mid = (x & (lowbitLen - 1)) + lowbitLen
            else:
                mid = (x & (lowbitLen - 1))
            applySwap(x, mid)
            applySwap(y, mid)
            applySwap(x, mid)
            return

        # 获取对应的目标位置
        pA, pB = getPairPos(A, B)
        pX, pY = getPairPos(x, y)

        # 核心共轭构造
        useAddMagic((pX - x + n) % n)
        useXorMagic(pX ^ pA)
        useAddMagic((A - pA + n) % n)

        useSwapMagic()

        useAddMagic((pA - A + n) % n)
        useXorMagic(pX ^ pA)
        useAddMagic((x - pX + n) % n)

    # 排列结构与分治处理
    class Permutation:
        def __init__(self):
            self.val = []
            self.len = 0
            self.ops = []

        def build(self):
            # C++ 源码中此处有一个必定为 true 的 vis 检查
            # 为了完全对齐满分代码的行为，不引入额外的 false 阻断，这里省略无用检查
            if self.len == 1:
                return True

            leftPart = Permutation()
            rightPart = Permutation()
            leftPart.len = self.len // 2
            rightPart.len = self.len // 2

            leftPart.val = [0] * leftPart.len
            rightPart.val = [0] * rightPart.len

            for i in range(self.len // 2):
                leftPart.val[i] = self.val[i * 2] // 2
                rightPart.val[i] = self.val[i * 2 + 1] // 2

            if not leftPart.build() or not rightPart.build():
                return False

            if self.val[0] & 1:
                self.ops.append(1 if self.len == 2 else -1)

            curXorL = 0
            for x in leftPart.ops:
                if x > 0:
                    self.ops.append(-1)
                    self.ops.append(1)
                else:
                    self.ops.append(x * 2)
                    curXorL ^= (-x) * 2
            
            if curXorL != 0:
                self.ops.append(-curXorL)

            curXorR = 0
            for x in rightPart.ops:
                if x > 0:
                    self.ops.append(1)
                    self.ops.append(-1)
                else:
                    self.ops.append(x * 2)
                    curXorR ^= (-x) * 2

            if (curXorR & (self.len // 2)) != (curXorL & (self.len // 2)):
                return False

            if curXorL >= self.len // 2:
                curXorL -= self.len // 2
            if curXorR >= self.len // 2:
                curXorR -= self.len // 2
            if curXorL != curXorR:
                return False

            # 合并魔法序列
            merged = []
            for x in self.ops:
                if not merged:
                    merged.append(x)
                else:
                    if x < 0 and merged[-1] < 0:
                        merged[-1] = -( (-merged[-1]) ^ (-x) )
                        if merged[-1] == 0:
                            merged.pop()
                    else:
                        merged.append(x)
            
            self.ops = merged
            return True

    # 主逻辑执行
    if lowbitLen > 1:
        base = Permutation()
        base.len = lowbitLen
        base.val = [0] * max(n, lowbitLen)
        for i in range(n):
            base.val[i] = p[i] & (lowbitLen - 1)

        if not base.build():
            print("-1")
            return

        for x in base.ops:
            if x > 0:
                useAddMagic(x)
            else:
                useXorMagic(-x)

    for rem in range(lowbitLen):
        vec = []
        for j in range(rem, n, lowbitLen):
            vec.append(p[j])

        vec.sort()

        ok = True
        ptr = 0
        for j in range(rem, n, lowbitLen):
            if vec[ptr] != j:
                ok = False
                break
            ptr += 1

        if not ok:
            print("-1")
            return

        for j in range(rem, n, lowbitLen):
            if p[j] != j:
                applySwap(j, p[j])

    # 最终结果输出
    print(len(answer))
    for x in answer:
        if x == 0:
            print("0")
        elif x < 0:
            print(f"1 {-x}")
        else:
            print(f"2 {x}")

if __name__ == '__main__':
    solve()

## B 长跑

In [ ]:
## add your code hereimport sys
from collections import deque

def solve():
    # 极速 IO 读取全部内容
    input_data = sys.stdin.read().split()
    if not input_data:
        return
    
    idx = 0
    total_len = len(input_data)
    out = []
    
    INF = 10**18 # 使用整型大数代替 float('inf') 提升比较速度
    
    while idx + 3 < total_len:
        N = int(input_data[idx])
        L = int(input_data[idx+1])
        Maxn = int(input_data[idx+2])
        S = int(input_data[idx+3])
        idx += 4
        
        stations = []
        for _ in range(N):
            p = int(input_data[idx])
            c = int(input_data[idx+1])
            idx += 2
            if p <= L:
                stations.append((p, c))
                
        # 按距离升序排序
        stations.sort(key=lambda x: x[0])
        
        all_stations = [(0, 0)] + stations
        n_all = len(all_stations)
        
        dp = [INF] * n_all
        dp[0] = 0
        
        # 单调队列，存储的是 all_stations 的索引
        # 队列维持两个性质：
        # 1. 索引从左到右递增，且距离在 Maxn 范围内
        # 2. 对应的 dp 值从左到右严格递增
        q = deque([0])
        
        for i in range(1, n_all):
            pi, ci = all_stations[i]
            
            # 1. 剔除队首已经超出 Maxn 距离范围的过时站点
            while q and pi - all_stations[q[0]][0] > Maxn:
                q.popleft()
                
            # 2. 此时队首 q[0] 就是合法滑动窗口内 dp 最小的那个！
            if q:
                dp[i] = dp[q[0]] + ci
                
            # 3. 维护单调队列的单调递增性
            # 如果当前站点的 dp 值比队尾的还要小（或相等），
            # 那么队尾的站点以后永远不可能成为最优选择，直接弹出
            if dp[i] != INF:
                while q and dp[q[-1]] >= dp[i]:
                    q.pop()
                q.append(i)
                
        # 统计一步能跨到终点的最小花费
        min_cost_to_end = INF
        for i in range(n_all):
            if L - all_stations[i][0] <= Maxn:
                if dp[i] < min_cost_to_end:
                    min_cost_to_end = dp[i]
                    
        if min_cost_to_end <= S:
            out.append("Yes")
        else:
            out.append("No")

    # 一次性快速输出所有用例结果
    sys.stdout.write('\n'.join(out) + '\n')

if __name__ == '__main__':
    solve()

## C 最长回文

In [ ]:
## add your code here#include <bits/stdc++.h>
using namespace std;

using i64 = long long;

const i64 P1 = 1000000007;
const i64 P2 = 1000000009;
const i64 X1 = 131;
const i64 X2 = 137;

vector<i64> q1, q2;

void init(int n) {
    q1.resize(n + 1);
    q2.resize(n + 1);

    q1[0] = q2[0] = 1;

    for (int i = 1; i <= n; i++) {
        q1[i] = q1[i - 1] * X1 % P1;
        q2[i] = q2[i - 1] * X2 % P2;
    }
}

struct H {
    vector<i64> a, b;

    H() {}

    H(const string& s) {
        make(s);
    }

    void make(const string& s) {
        int n = (int)s.size();

        a.assign(n + 1, 0);
        b.assign(n + 1, 0);

        for (int i = 0; i < n; i++) {
            int c = s[i];

            a[i + 1] = (a[i] * X1 + c) % P1;
            b[i + 1] = (b[i] * X2 + c) % P2;
        }
    }

    pair<i64, i64> ask(int l, int r) const {
        if (l > r) return {0, 0};

        int len = r - l + 1;

        i64 u = (a[r] - a[l - 1] * q1[len] % P1 + P1) % P1;
        i64 v = (b[r] - b[l - 1] * q2[len] % P2 + P2) % P2;

        return {u, v};
    }
};

vector<int> work(const string& s) {
    string t;
    t.reserve(s.size() * 2 + 1);

    for (char c : s) {
        t += '#';
        t += c;
    }
    t += '#';

    int m = (int)t.size();
    vector<int> r(m);

    int p = 0, mx = 0;

    for (int i = 0; i < m; i++) {
        if (i < mx) {
            int j = p * 2 - i;
            r[i] = min(mx - i, r[j]);
        }

        while (
            i - r[i] - 1 >= 0 &&
            i + r[i] + 1 < m &&
            t[i - r[i] - 1] == t[i + r[i] + 1]
        ) {
            r[i]++;
        }

        if (i + r[i] > mx) {
            p = i;
            mx = i + r[i];
        }
    }

    return r;
}

int get(
    int x,
    int y,
    int n,
    const H& A,
    const H& B
) {
    if (x <= 0 || x > n || y <= 0 || y > n) {
        return 0;
    }

    int l1 = n - x + 1;
    int l2 = y;

    int L = 1;
    int R = min(x, n - y + 1);
    int ans = 0;

    while (L <= R) {
        int mid = (L + R) >> 1;

        if (A.ask(l1, l1 + mid - 1) == B.ask(l2, l2 + mid - 1)) {
            ans = mid;
            L = mid + 1;
        } else {
            R = mid - 1;
        }
    }

    return ans;
}

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    int n;

    while (cin >> n) {
        string s, t;
        cin >> s >> t;

        init(n);

        string z = s;
        reverse(z.begin(), z.end());

        H A(z), B(t);

        vector<int> rs = work(s);
        vector<int> rt = work(t);

        int ans = 0;

        for (int i = 0; i <= 2 * n; i++) {
            int L = (i - rs[i]) / 2 + 1;
            int R = (i + rs[i]) / 2;

            int x = L - 1;
            int y = R;

            int k = get(x, y, n, A, B);

            ans = max(ans, rs[i] + k * 2);
        }

        for (int i = 0; i <= 2 * n; i++) {
            int L = (i - rt[i]) / 2 + 1;
            int R = (i + rt[i]) / 2;

            int x = L;
            int y = R + 1;

            int k = get(x, y, n, A, B);

            ans = max(ans, rt[i] + k * 2);
        }

        cout << ans << '\n';
    }

    return 0;
}

## D 优惠券

In [ ]:
## add your code here#include <bits/stdc++.h>
using namespace std;

const int M = 100005;

int a[M], b[M];

bool check(
    int x,
    int val,
    int pos,
    set<int>& box,
    vector<int>& used,
    int& bad
) {
    if (a[x] == val) {
        auto it = box.upper_bound(b[x]);

        if (it == box.end()) {
            bad = pos;
            return false;
        }

        box.erase(it);
        b[x] = pos;
        used.push_back(x);
    } else {
        a[x] = val;
        b[x] = pos;
        used.push_back(x);
    }

    return true;
}

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    int n;

    while (cin >> n) {
        set<int> s;
        vector<int> v;

        int ans = -1;
        bool flag = true;

        for (int i = 1; i <= n; i++) {
            string op;
            cin >> op;

            int x = -1;

            if (op == "I" || op == "O") {
                cin >> x;
            }

            if (!flag) {
                continue;
            }

            if (op == "?") {
                s.insert(i);
            } else if (op == "I") {
                if (!check(x, 1, i, s, v, ans)) {
                    flag = false;
                }
            } else if (op == "O") {
                if (!check(x, 0, i, s, v, ans)) {
                    flag = false;
                }
            } else {
                s.insert(i);
            }
        }

        for (int x : v) {
            a[x] = 0;
            b[x] = 0;
        }

        cout << ans << '\n';
    }

    return 0;
}

## E 任意点

In [ ]:
## add your code here#include <bits/stdc++.h>
using namespace std;

struct DSU {
    vector<int> f;

    DSU(int n = 0) {
        f.resize(n);
        iota(f.begin(), f.end(), 0);
    }

    int find(int x) {
        if (f[x] == x) return x;
        return f[x] = find(f[x]);
    }

    void merge(int x, int y) {
        int fx = find(x);
        int fy = find(y);

        if (fx != fy) {
            f[fx] = fy;
        }
    }
};

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    int n;
    cin >> n;

    vector<int> x(n), y(n);

    for (int i = 0; i < n; i++) {
        cin >> x[i] >> y[i];
    }

    DSU d(n);

    for (int i = 0; i < n; i++) {
        for (int j = i + 1; j < n; j++) {
            if (x[i] == x[j] || y[i] == y[j]) {
                d.merge(i, j);
            }
        }
    }

    set<int> s;

    for (int i = 0; i < n; i++) {
        s.insert(d.find(i));
    }

    cout << (int)s.size() - 1 << '\n';

    return 0;
}

## F 通配符匹配

In [ ]:
## add your code here#include <bits/stdc++.h>
using namespace std;

using ull = unsigned long long;

const ull B = 131;
const int N = 100000 + 5;

vector<ull> pw;

ull getVal(const string& s) {
    ull res = 0;
    for (char c : s) {
        res = res * B + c;
    }
    return res;
}

struct Hash {
    vector<ull> h;

    Hash() {}

    Hash(const string& s) {
        build(s);
    }

    void build(const string& s) {
        int n = s.size();
        h.assign(n + 1, 0);

        for (int i = 0; i < n; i++) {
            h[i + 1] = h[i] * B + s[i];
        }
    }

    ull ask(int l, int len) const {
        return h[l + len] - h[l] * pw[len];
    }
};

bool samePart(
    const Hash& hs,
    int pos,
    const string& p,
    ull val,
    int lim
) {
    int len = p.size();

    if (pos + len > lim) {
        return false;
    }

    return hs.ask(pos, len) == val;
}

bool judge(
    const string& s,
    const vector<string>& part,
    const vector<char>& mark,
    const vector<ull>& hv
) {
    int n = s.size();
    int m = part.size();

    Hash hs(s);

    vector<vector<char>> dp(m + 1, vector<char>(n + 1, 0));
    vector<vector<char>> go(m + 1, vector<char>(n + 1, 0));

    dp[0][0] = 1;

    for (int pos = 0; pos < n; pos++) {
        for (int id = 0; id <= m; id++) {
            if (go[id][pos]) {
                dp[id][pos + 1] = 1;
                go[id][pos + 1] = 1;
            }

            if (id == m || !dp[id][pos]) {
                continue;
            }

            int len = part[id].size();

            if (!samePart(hs, pos, part[id], hv[id], n)) {
                continue;
            }

            int nxt = pos + len;

            if (mark[id] == '?') {
                if (nxt + 1 <= n) {
                    dp[id + 1][nxt + 1] = 1;
                }
            } else {
                dp[id + 1][nxt] = 1;
                go[id + 1][nxt] = 1;
            }
        }
    }

    return dp[m][n];
}

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    pw.assign(N, 1);
    for (int i = 1; i < N; i++) {
        pw[i] = pw[i - 1] * B;
    }

    string pat;
    getline(cin, pat);

    vector<string> part;
    vector<char> mark;

    string cur;

    for (char c : pat) {
        if (c == '*' || c == '?') {
            part.push_back(cur);
            mark.push_back(c);
            cur.clear();
        } else {
            cur += c;
        }
    }

    part.push_back(cur);
    mark.push_back('?');

    vector<ull> hv(part.size());

    for (int i = 0; i < (int)part.size(); i++) {
        hv[i] = getVal(part[i]);
    }

    int T;
    cin >> T;

    while (T--) {
        string s;
        cin >> s;

        s += '#';

        if (judge(s, part, mark, hv)) {
            cout << "YES\n";
        } else {
            cout << "NO\n";
        }
    }

    return 0;
}

## G 汉诺塔

In [ ]:
## add your code here#include <cstdio>
#include <cmath>

using ll = long long;

char moveName[4];
int order[9];
int diskCount;
int resultType;
int remain = 6;

ll calculateAnswer(int type) {
    if (type == 1) {
        return (ll)2 * pow(3, diskCount - 1) - 1;
    }

    if (type) {
        return (ll)pow(2, diskCount) - 1;
    }

    return (ll)pow(3, diskCount - 1);
}

int main() {
    scanf("%d", &diskCount);

    while (remain--) {
        scanf("%s", moveName);
        order[(moveName[0] - 'A') * 3 + moveName[1] - 'A'] = remain;
    }

    if (order[1] > order[2]) {
        if (order[5] < order[3]) {
            resultType = 1;
        } else if (order[6] > order[7]) {
            resultType = 2;
        }
    } else if (order[7] < order[6]) {
        resultType = 1;
    } else if (order[3] > order[5]) {
        resultType = 2;
    }

    printf("%lld", calculateAnswer(resultType));

    return 0;
}

## H 马步距离

In [ ]:
## add your code here#include <bits/stdc++.h>
using namespace std;

int main() {
    long long startX, startY, targetX, targetY;
    cin >> startX >> startY >> targetX >> targetY;

    long long diffX = abs(startX - targetX);
    long long diffY = abs(startY - targetY);

    if (diffX < diffY) {
        swap(diffX, diffY);
    }

    if (diffX == 1 && diffY == 0) {
        cout << 3 << '\n';
        return 0;
    }

    if (diffX == 2 && diffY == 2) {
        cout << 4 << '\n';
        return 0;
    }

    long long steps = max((diffX + 1) / 2, (diffX + diffY + 2) / 3);

    if ((steps & 1) != ((diffX + diffY) & 1)) {
        ++steps;
    }

    cout << steps << '\n';

    return 0;
}

## I 直方图最大矩形

In [ ]:
## add your code hereclass Solution {
public:
    int largestRectangleArea(vector<int>& h) {
        int n = h.size();

        vector<int> l(n), r(n, n);
        vector<int> st;

        for (int i = 0; i < n; i++) {
            while (!st.empty() && h[st.back()] >= h[i]) {
                st.pop_back();
            }

            if (st.empty()) {
                l[i] = -1;
            } else {
                l[i] = st.back();
            }

            st.push_back(i);
        }

        st.clear();

        for (int i = n - 1; i >= 0; i--) {
            while (!st.empty() && h[st.back()] >= h[i]) {
                st.pop_back();
            }

            if (st.empty()) {
                r[i] = n;
            } else {
                r[i] = st.back();
            }

            st.push_back(i);
        }

        long long ans = 0;

        for (int i = 0; i < n; i++) {
            long long w = r[i] - l[i] - 1;
            ans = max(ans, w * h[i]);
        }

        return (int)ans;
    }
};

## J 消防局的设立

In [ ]:
## add your code here#include <bits/stdc++.h>
using namespace std;

using ll = long long;

const ll BIG = 4e18;

vector<vector<int>> g;
vector<array<ll, 5>> f;

void run(int u, int fa) {
    int cnt = 0;
    ll s2 = 0, s3 = 0;

    for (int v : g[u]) {
        if (v == fa) continue;

        run(v, u);

        s2 += f[v][2];
        s3 += f[v][3];
        cnt++;
    }

    if (cnt == 0) {
        f[u][0] = 1;
        f[u][1] = 1;
        f[u][2] = 1;
        f[u][3] = 0;
        f[u][4] = 0;
        return;
    }

    f[u][0] = 1;
    f[u][1] = BIG;
    f[u][2] = BIG;
    f[u][3] = 0;
    f[u][4] = 0;

    for (int v : g[u]) {
        if (v == fa) continue;

        f[u][0] += f[v][4];

        f[u][1] = min(
            f[u][1],
            f[v][0] + s3 - f[v][3]
        );

        f[u][2] = min(
            f[u][2],
            f[v][1] + s2 - f[v][2]
        );

        f[u][3] += f[v][2];
        f[u][4] += f[v][3];
    }

    for (int i = 1; i < 5; i++) {
        f[u][i] = min(f[u][i], f[u][i - 1]);
    }
}

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    int n;
    cin >> n;

    g.assign(n + 1, {});
    f.assign(n + 1, {});

    for (int i = 2; i <= n; i++) {
        int p;
        cin >> p;

        g[i].push_back(p);
        g[p].push_back(i);
    }

    run(1, 0);

    cout << f[1][2] << '\n';

    return 0;
}